### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp9
Data variations: ['none']
Has transformation: True
Threshold corr:	 0.5
Groups id:	 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']
Subgroups id:	 {'AR': ['1', '2'], 'CRS': ['1', '2'], 'OSA': ['1', '2'], 'LPRD': ['1', '2'], 'SGB': ['1', '2'], 'LSNB': ['1', '2'], 'RCC': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


In [4]:
# Remove
# groups_id = ["OSA"]

### Load dataset

In [5]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,1.0,69.99951,Unknown,3.761221,0.006085,0.013273,3.028000e-02,0.039917,0.050244,0.061226,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,1.0,70.04025,Unknown,330.900909,3.345932,7.288550,1.660211e+01,21.875132,27.522388,33.526282,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,1.0,70.04151,Unknown,3.810500,0.323930,0.609772,1.190270e+00,1.489259,1.794746,2.106842,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,1.0,70.04908,Unknown,371.967656,0.779912,1.640943,3.603099e+00,4.689464,5.839993,7.051657,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,1.0,70.06267,Unknown,14.383827,0.705709,1.341339,1.985815e+00,2.645173,3.320973,4.013620,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,163.972145,0.257934,0.408042,6.627189e-01,0.779649,17223.650862,0.892598,...,1.267927,1.498143,2.841460,1.518527,2.825464,1.477947,2.080693,1.528791,2.042619,1.488022
5440,1.0,748.76437,Unknown,5.318578,0.037040,0.088541,2.224989e-01,0.302977,0.391783,0.488620,...,1.576491,1.885946,3.803859,1.913544,3.757898,1.858634,2.703381,1.927451,2.650225,1.872254
5441,1.0,794.79590,Unknown,44.795976,0.062477,0.158303,4.230803e-01,0.588121,0.773688,0.979274,...,2.081027,2.412741,4.164254,2.447782,4.113120,2.378048,421.344464,2.465433,3.246804,2.395351
5442,1.0,800.81295,Unknown,32.469955,0.091943,0.246783,7.009899e-01,0.994539,1.330769,1.709164,...,5.679157,6.495188,10.681201,40.563104,10.573695,6.418109,8.392011,6.572858,8.300825,6.456575


In [6]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,1.0,69.99951
1,1.0,70.04025
2,1.0,70.04151
3,1.0,70.04908
4,1.0,70.06267
...,...,...
5439,1.0,732.79951
5440,1.0,748.76437
5441,1.0,794.79590
5442,1.0,800.81295


In [7]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,3.761221,0.006085,0.013273,3.028000e-02,0.039917,0.050244,0.061226,0.072841,0.125292,0.170628,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,330.900909,3.345932,7.288550,1.660211e+01,21.875132,27.522388,33.526282,39.873536,68.517547,93.257600,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,3.810500,0.323930,0.609772,1.190270e+00,1.489259,1.794746,2.106842,2.425556,3.765647,4.837455,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,371.967656,0.779912,1.640943,3.603099e+00,4.689464,5.839993,7.051657,8.322116,13.959503,18.740646,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,14.383827,0.705709,1.341339,1.985815e+00,2.645173,3.320973,4.013620,4.723118,7.726113,10.145765,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,163.972145,0.257934,0.408042,6.627189e-01,0.779649,17223.650862,0.892598,1.002637,1.321292,1.629611,...,1.267927,1.498143,2.841460,1.518527,2.825464,1.477947,2.080693,1.528791,2.042619,1.488022
5440,5.318578,0.037040,0.088541,2.224989e-01,0.302977,0.391783,0.488620,0.593275,1.087500,1.535636,...,1.576491,1.885946,3.803859,1.913544,3.757898,1.858634,2.703381,1.927451,2.650225,1.872254
5441,44.795976,0.062477,0.158303,4.230803e-01,0.588121,0.773688,0.979274,1.204545,2.299264,2.962123,...,2.081027,2.412741,4.164254,2.447782,4.113120,2.378048,421.344464,2.465433,3.246804,2.395351
5442,32.469955,0.091943,0.246783,7.009899e-01,0.994539,1.330769,1.709164,2.129490,4.231004,6.255025,...,5.679157,6.495188,10.681201,40.563104,10.573695,6.418109,8.392011,6.572858,8.300825,6.456575


In [8]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5444 entries, 0 to 5443
Columns: 1248 entries, AR_1.1 to PD_2.52
dtypes: float64(1248)
memory usage: 51.9 MB


In [9]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 6794112


### Generate graphs

In [10]:
""" from sklearn import preprocessing

X_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)

df_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)
df_join_raw_log """

' from sklearn import preprocessing\n\nX_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)\n\ndf_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)\ndf_join_raw_log '

In [11]:
# Transformation (log10)

if not has_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,3.761221,0.006085,0.013273,0.030280,0.039917,0.050244,0.061226,0.072841,0.125292,0.170628,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,330.900909,3.345932,7.288550,16.602111,21.875132,27.522388,33.526282,39.873536,68.517547,93.257600,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,3.810500,0.323930,0.609772,1.190270,1.489259,1.794746,2.106842,2.425556,3.765647,4.837455,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,371.967656,0.779912,1.640943,3.603099,4.689464,5.839993,7.051657,8.322116,13.959503,18.740646,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,14.383827,0.705709,1.341339,1.985815,2.645173,3.320973,4.013620,4.723118,7.726113,10.145765,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638


In [12]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 6794112


In [13]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,AR_1.221,AR_1.222,AR_1.223,AR_1.224,AR_1.225,AR_1.226,AR_1.227,AR_1.228,AR_1.229,AR_1.230
0,3.761221,0.006085,0.013273,0.030280,0.039917,0.050244,0.061226,0.072841,0.125292,0.170628,...,0.115679,2.591635,0.116000,0.117606,0.118249,3.152372,2.739992,5.079413,7.373953,2.656724
1,330.900909,3.345932,7.288550,16.602111,21.875132,27.522388,33.526282,39.873536,68.517547,93.257600,...,167.035094,168.051756,173.202682,183.848824,186.034213,188.238621,193.833710,196.105717,198.397337,199.550543
2,3.810500,0.323930,0.609772,1.190270,1.489259,1.794746,2.106842,2.425556,3.765647,4.837455,...,61.398011,3.468835,48.126401,3.538710,3.545692,45.525540,61.387017,3.566630,45.540835,44.316094
3,371.967656,0.779912,1.640943,3.603099,4.689464,5.839993,7.051657,8.322116,13.959503,18.740646,...,5.657818,72.845627,5.690376,5.706667,17.329060,419.508552,21.376463,219.669060,38.173112,212.791064
4,14.383827,0.705709,1.341339,1.985815,2.645173,3.320973,4.013620,4.723118,7.726113,10.145765,...,12.064185,12.090685,12.223326,29.969955,12.436053,588.303200,12.542658,12.596023,12.622721,12.649430


In [14]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1252120


In [15]:
# Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)
dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,3.761221,330.900909,3.810500,371.967656,14.383827,18.712536,417.143534,436.417402,447.090337,546.476338,...,44.344399,97.695020,0.490535,17.259178,74.363399,163.972145,5.318578,44.795976,32.469955,8.757156e+01
1,0.006085,3.345932,0.323930,0.779912,0.705709,1.268857,2.415192,2.922833,2.590675,3.762715,...,1.133275,0.837680,0.009950,0.498313,0.545725,0.257934,0.037040,0.062477,0.091943,1.610846e-02
2,0.013273,7.288550,0.609772,1.640943,1.341339,2.303849,4.318308,5.183555,4.703248,6.515885,...,2.634712,1.662149,0.023167,1.027306,0.918779,0.408042,0.088541,0.158303,0.246783,3.576926e-02
3,0.030280,16.602111,1.190270,3.603099,1.985815,4.328700,7.982784,9.500080,8.835714,11.644681,...,6.429114,3.430323,0.056623,2.207620,1.593778,0.662719,0.222499,0.423080,0.700990,1.076772e+06
4,0.039917,21.875132,1.489259,4.689464,2.645173,5.347228,9.807522,11.638008,10.914225,14.145286,...,8.668720,4.372835,0.076389,2.852567,1.916802,0.779649,0.302977,0.588121,0.994539,8.314640e-02


In [16]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1252120


In [17]:
# Correlation matrix

dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

AR 1 (230, 5444)


AR 2 (230, 5444)
CRS 1 (14, 5444)
CRS 2 (14, 5444)


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/pingouin/correlation.py:954: RuntimeWarning: invalid value encountered in sqrt
  D = np.diag(np.sqrt(1 / np.diag(Vi)))


OSA 1 (29, 5444)
OSA 2 (30, 5444)
LPRD 1 (16, 5444)
LPRD 2 (16, 5444)
SGB 1 (43, 5444)
SGB 2 (44, 5444)
LSNB 1 (34, 5444)
LSNB 2 (35, 5444)
RCC 1 (71, 5444)
RCC 2 (72, 5444)
BC 1 (41, 5444)
BC 2 (42, 5444)
BPH 1 (39, 5444)
BPH 2 (39, 5444)
PCa 1 (53, 5444)
PCa 2 (53, 5444)
PD 1 (51, 5444)
PD 2 (52, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,-0.000263,0.000323,0.000319,-0.000306,-0.000372,-0.000628,0.000112,-0.000132,0.000371,...,-0.074758,0.164191,0.004363,-0.002529,0.004560,-0.036402,0.042348,0.000487,-0.004211,0.002699
1,-0.000263,1.000000,1.000000,1.000000,-1.000000,-1.000000,-0.999994,-1.000000,-1.000000,1.000000,...,0.957228,-0.788030,-0.997870,-0.995818,-0.998317,0.951918,0.947874,0.997395,-0.998506,-0.999291
2,0.000323,1.000000,1.000000,-1.000000,1.000000,1.000000,0.999994,0.999999,1.000000,-1.000000,...,-0.957223,0.788003,0.997873,0.995823,0.998321,-0.951922,-0.947868,-0.997398,0.998509,0.999294
3,0.000319,1.000000,-1.000000,1.000000,1.000000,1.000000,0.999994,0.999999,1.000000,-1.000000,...,-0.957222,0.788005,0.997873,0.995823,0.998321,-0.951921,-0.947866,-0.997399,0.998509,0.999294
4,-0.000306,-1.000000,1.000000,1.000000,1.000000,-1.000000,-0.999994,-1.000000,-1.000000,1.000000,...,0.957229,-0.788016,-0.997869,-0.995822,-0.998319,0.951905,0.947873,0.997395,-0.998506,-0.999292


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'AR': {'1':           0         1         2         3         4         5         6     \
  0     1.000000 -0.000263  0.000323  0.000319 -0.000306 -0.000372 -0.000628   
  1    -0.000263  1.000000  1.000000  1.000000 -1.000000 -1.000000 -0.999994   
  2     0.000323  1.000000  1.000000 -1.000000  1.000000  1.000000  0.999994   
  3     0.000319  1.000000 -1.000000  1.000000  1.000000  1.000000  0.999994   
  4    -0.000306 -1.000000  1.000000  1.000000  1.000000 -1.000000 -0.999994   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439 -0.036402  0.951918 -0.951922 -0.951921  0.951905  0.951873  0.951912   
  5440  0.042348  0.947874 -0.947868 -0.947866  0.947873  0.947872  0.947816   
  5441  0.000487  0.997395 -0.997398 -0.997399  0.997395  0.997391  0.997439   
  5442 -0.004211 -0.998506  0.998509  0.998509 -0.998506 -0.998502 -0.998506   
  5443  0.002699 -0.999291  0.999294  0.999294 -0.999292 -0.999287 -0.999297   
  
            7         8   

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 15052776
Count zero:	 0
Count positive:	 14584360


In [20]:
# Build graph (corpus graphs)

dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
3547,0,3548,0.586009,1
3639,0,3640,-0.595230,1
5443,1,2,1.000000,1
5444,1,3,1.000000,1
5445,1,4,-1.000000,1


In [21]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [22]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

  0%|          | 0/11 [00:00<?, ?it/s]

100%|██████████| 11/11 [07:22<00:00, 40.18s/it]


In [23]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,AR,1,5426,13817755,0.938832,NaN,False
1,AR,2,5367,107977,0.007499,NaN,False
2,CRS,1,5444,4870424,0.328731,NaN,True
3,CRS,2,3521,6196960,1.000000,NaN,True
4,OSA,1,5444,4952086,0.334243,NaN,True
5,OSA,2,5443,3641622,0.245883,NaN,True
6,LPRD,1,4035,8138595,1.000000,NaN,True
7,LPRD,2,5444,8015568,0.541013,NaN,False
8,SGB,1,5442,2102654,0.142024,NaN,True
9,SGB,2,2,1,1.000000,NaN,True


---

In [24]:
X = dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]]
X

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,3.761221,330.900909,3.810500,371.967656,14.383827,18.712536,417.143534,436.417402,447.090337,546.476338,...,44.344399,97.695020,0.490535,17.259178,74.363399,163.972145,5.318578,44.795976,32.469955,8.757156e+01
1,0.006085,3.345932,0.323930,0.779912,0.705709,1.268857,2.415192,2.922833,2.590675,3.762715,...,1.133275,0.837680,0.009950,0.498313,0.545725,0.257934,0.037040,0.062477,0.091943,1.610846e-02
2,0.013273,7.288550,0.609772,1.640943,1.341339,2.303849,4.318308,5.183555,4.703248,6.515885,...,2.634712,1.662149,0.023167,1.027306,0.918779,0.408042,0.088541,0.158303,0.246783,3.576926e-02
3,0.030280,16.602111,1.190270,3.603099,1.985815,4.328700,7.982784,9.500080,8.835714,11.644681,...,6.429114,3.430323,0.056623,2.207620,1.593778,0.662719,0.222499,0.423080,0.700990,1.076772e+06
4,0.039917,21.875132,1.489259,4.689464,2.645173,5.347228,9.807522,11.638008,10.914225,14.145286,...,8.668720,4.372835,0.076389,2.852567,1.916802,0.779649,0.302977,0.588121,0.994539,8.314640e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,3.152372,188.238621,45.525540,419.508552,588.303200,632.905658,868.230825,893.355253,891.266566,1017.304303,...,2266.695130,789.492613,0.353199,125.045388,813.632647,2.595428,3.301592,717.376519,526.243502,2.091185e+02
226,2.739992,193.833710,61.387017,21.376463,12.542658,16.362260,26.896021,27.274930,24.390295,24.720117,...,126.410739,20.395373,26.030318,689.883993,5.830708,2.671095,3.405531,88.003434,10.043272,9.183591e-01
227,5.079413,196.105717,3.566630,219.669060,12.596023,16.428755,298.451725,329.900933,324.857513,422.506955,...,36.772413,20.607614,0.358340,13.005563,5.900927,2.686421,3.447796,3.938186,10.147011,9.288274e-01
228,7.373953,198.397337,45.540835,38.173112,12.622721,16.462014,26.947281,27.381806,24.489886,28.985015,...,36.970233,20.821374,0.360064,13.118376,5.936268,2.717266,3.490459,3.962902,10.251487,9.393837e-01


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[ 0.96013323, -0.18319677, -0.06898287, ..., -0.11258906,
        -0.07363153, -0.07805489],
       [-0.91476946, -0.21063439, -0.06945523, ..., -0.11268725,
        -0.07365321, -0.07926111],
       [-0.91118024, -0.21030414, -0.06941651, ..., -0.11268704,
        -0.07365311, -0.07926084],
       ...,
       [ 1.61829342, -0.19448788, -0.06901591, ..., -0.11267875,
        -0.07364648, -0.07924854],
       [ 2.76393481, -0.19429592, -0.06332917, ..., -0.11267869,
        -0.07364641, -0.07924839],
       [ 0.40866874, -0.19419933, -0.0634951 , ..., -0.11267864,
        -0.07364637, -0.07924832]])

In [26]:
import numpy as np
from sklearn.covariance import GraphicalLassoCV

cv = min(5, X_scaled.shape[0])

# X_scaled: matriz (n_muestras, p_metabolitos), ya estandarizada (media 0, var 1)
modelo = GraphicalLassoCV(cv=3, n_jobs=-1) # cv=3, max_iter=200)
modelo.fit(X_scaled)

Omega = modelo.precision_          # matriz de precisión dispersa (p x p)
lambda_optimo = modelo.alpha_      # lambda elegido por CV

# Correlación parcial a partir de Omega
d = np.sqrt(np.diag(Omega))
P = -Omega / np.outer(d, d)
np.fill_diagonal(P, 1.0)           # P[i,j] = correlación parcial entre i y j

""" # Matriz de adyacencia binaria para tu grafo (umbral en 0, ya que glasso fuerza ceros exactos)
adyacencia = (Omega != 0).astype(int)
np.fill_diagonal(adyacencia, 0)
adyacencia """
P

/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/sklearn/covariance/_graph_lasso.py:139: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.007372212094168162, tolerance: 0.006431368037763544
  coefs, _, _, _ = cd_fast.enet_coordinate_descent_gram(
/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/sklearn/covariance/_graph_lasso.py:139: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.008086189842913427, tolerance: 0.007031509353046799
  coefs, _, _, _ = cd_fast.enet_coordinate_descent_gram(
/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/sklearn/covariance/_graph_lasso.py:139: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.04142898136007034, tolerance: 0.02406804804190885
  coefs, _, _, _ = cd_fast.enet_coordin

KeyboardInterrupt: 